[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/terbe2022/RIMS-Archival-Project-/blob/main/notebooks/00_environment_check.ipynb)

# 00 — Environment Check

Run this first, every new Colab session. It clones the repo, installs dependencies,
reports what hardware you actually got, and confirms the Box connection works.

**Why the repo gets cloned rather than the code being pasted here:** the pipeline logic
lives in `src/` as ordinary Python modules. Notebooks import it. That way there is one
copy of every function, it can be diffed and reviewed like normal code, and the same code
that runs here runs on the L4 server. Notebooks are for explaining and for driving a GPU —
not for holding the logic.


## 1. Clone the repo

Colab's filesystem is wiped when the runtime recycles, so this runs fresh every session.
The repo is public, so no token is needed.


In [ ]:
!git clone -q https://github.com/terbe2022/RIMS-Archival-Project-.git
%cd RIMS-Archival-Project-
!git log --oneline -3


## 2. Install dependencies

Takes a couple of minutes. Pinned in the repo so every session is identical.


In [ ]:
!pip install -q box-sdk-gen pandas pyarrow python-dotenv sentence-transformers
print('done')


## 3. What hardware did we actually get?

Colab's free tier gives you a T4 *when one is available*. Never assume — check.
If this shows no GPU: Runtime → Change runtime type → T4 GPU, then re-run.


In [ ]:
import subprocess, torch

gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu.returncode == 0:
    print(gpu.stdout.split('\n')[8])   # the row with the GPU name and memory
    print(f'torch sees CUDA: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f'{p.name}, {p.total_memory/1e9:.1f} GB')
else:
    print('No GPU on this runtime.')
    print('Runtime -> Change runtime type -> T4 GPU')


## 4. Box credentials

**Never paste a token into a cell.** It gets saved into the notebook file, and if that is
committed it is in the public repo history permanently.

Use Colab's secrets manager instead — the **key icon** in the left sidebar:

1. Add a secret named `BOX_DEVELOPER_TOKEN`
2. Paste the token from the Box Developer Console as the value
3. Toggle **Notebook access** on

Secrets live with your Google account, never in the file.


In [ ]:
from google.colab import userdata
import os

try:
    os.environ['BOX_DEVELOPER_TOKEN'] = userdata.get('BOX_DEVELOPER_TOKEN')
    print('Box token loaded from Colab secrets')
except Exception as e:
    print('No BOX_DEVELOPER_TOKEN secret found.')
    print('Key icon in the left sidebar -> Add new secret -> enable notebook access.')

os.environ['BOX_ROOT_FOLDER_ID'] = '318353711369'   # TomHanratty accession


## 5. Test the Box connection

Imports the same `client_from_env()` the command-line scripts use. It picks up the
developer token automatically and falls back to the Service Account when the app is
authorized — so nothing here changes when that happens.


In [ ]:
import sys; sys.path.insert(0, 'src')
from box_inventory import client_from_env

client = client_from_env()
me = client.users.get_user_me()
print(f'authenticated as: {me.name} ({me.login})')

folder = client.folders.get_folder_by_id(os.environ['BOX_ROOT_FOLDER_ID'])
print(f'folder: {folder.name}')


## 6. Peek at the folder

One page of items, metadata only. Note that `sha1` comes back without downloading
anything — that is what makes deduplication free and why triage can run before any
transfer happens.


In [ ]:
import pandas as pd

items = client.folders.get_folder_items(
    os.environ['BOX_ROOT_FOLDER_ID'], limit=25, usemarker=True,
    fields=['id','type','name','size','sha1','extension','modified_at'])

rows = [{'name': i.name, 'type': i.type,
         'size_mb': round((getattr(i,'size',0) or 0)/1e6, 2),
         'ext': getattr(i,'extension','') or '',
         'sha1': (getattr(i,'sha1','') or '')[:12]}
        for i in items.entries]

df = pd.DataFrame(rows)
print(f'{len(df)} items on the first page')
df.head(25)


## 7. Persist anything you want to keep

`/content` is wiped when the runtime recycles — on disconnect, on idle timeout, and
always within about 12 hours. Mount Drive for anything that has to survive.

Never write accession content to Drive. Manifests and results only.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUT = '/content/drive/MyDrive/rims/manifest'
os.makedirs(OUT, exist_ok=True)
print(f'outputs -> {OUT}')


---

## If everything above ran

You are set up. Next: `01_box_inventory.ipynb`.

**Before saving this notebook back to GitHub:**

1. Edit → **Clear all outputs** — outputs bloat the repo and can leak real content
2. File → Save a copy in GitHub
3. Commit to a **branch**, not `main` — e.g. `gauri/setup` — then open a pull request

One of the POC notebooks was 377 MB because nobody cleared outputs. Worth the two seconds.
